In [1]:
import numpy as np
# каждые X точек из K в каждый кластер M
# 
import random

# =========================
# ЧТЕНИЕ ДАННЫХ
# =========================
# N — número de datos (cantidad de puntos)
# K — número de atributos (dimensión de cada punto)
# M — número de clusters (grupos)

# N — количество объектов (размер набора данных)
# K — размерность данных (число признаков у каждой точки)
# M — число кластеров, на которые делим данные
def leer_datos(fichero):
    with open(fichero, 'r') as f:
        linea = f.readline().split()
        N = int(linea[0])   # число точек
        K = int(linea[1])   # размерность
        M = int(linea[2])   # число кластеров
        # número de puntos / cantidad de puntos
        # dimensionalidad / dimensión
        # número de clusters / número de grupos
        X = [] 


           

        for _ in range(N):            X.append(list(map(float, f.readline().split())))

        return np.array(X), N, K, M

In [2]:

# =========================
# ВЫЧИСЛЕНИЕ ЦЕНТРОВ
# =========================
# вычисляет центры кластеров на основе разбиения P и данных X
# =========================
# CÁLCULO DE CENTROS
# =========================
# calcula los centros de los clusters basados en la partición P y los datos X
def calcular_medoides(P, X, M):
    """
    P - разбиение (вектор кластеров)
    X - данные
    M - число кластеров
    
    Возвращает medoids (представители кластеров из X)
    
    ---
    
    P - partición (vector de clusters)
    X - datos
    M - número de clusters
    
    Retorna medoids (representantes de clusters de X)
    """
    
    K = X.shape[1]
    medoids = np.zeros((M, K))
    
    for j in range(M):
        # выбираем точки кластера j
        puntos = X[P == j]
        
        if len(puntos) > 0:
            # считаем сумму расстояний для каждой точки
            mejor_punto = None
            mejor_dist = float('inf')
            
            for p in puntos:
                suma = 0.0
                
                for q in puntos:
                    suma += np.linalg.norm(p - q)
                
                # ищем точку с минимальной суммой расстояний
                if suma < mejor_dist:
                    mejor_dist = suma
                    mejor_punto = p
            
            medoids[j] = mejor_punto
        
        else:
            # если кластер пуст — случайная точка
            medoids[j] = X[random.randint(0, len(X)-1)]
    
    return medoids


In [3]:

# =========================
# FITNESS (ЦЕЛЕВАЯ ФУНКЦИЯ)
# =========================
# FITNESS (FUNCIÓN OBJETIVO)
# =========================

def fitness(P, X, M):
    # получаем центр каждого кластера
    # obtenemos el centro de cada cluster
    medoids = calcular_medoides(P, X, M)
    
    total = 0.0
    # считаем сумму расстояний` от каждой точки до центра её кластера`
    # calculamos la suma de distancias de cada punto al centro de su cluster
    for i in range(len(X)):
        c = medoids[P[i]]
        total += np.linalg.norm(X[i] - c)**2
    
    return total

In [4]:

# =========================
# ИНИЦИАЛИЗАЦИЯ ПОПУЛЯЦИИ
# =========================
# INICIALIZACIÓN DE LA POBLACIÓN
# =========================
# def inicializar_poblacion(popSize, N, M):
#     poblacion = []
    # в первом распрделении не допустить ситуации где нет
    # check 20р раз с разными сидами и 
    # set_random_seed(42)  # для воспроизводимости
    # разница между кластерами в бинарной ситуации 
    # 7 test results
    # также 
    # Создаем количество возможных решений, равное размеру популяции
    # Creamos la cantidad de posibles soluciones igual al tamaño de la población
def inicializar_poblacion(popSize, N, M):
    poblacion = []
    
    # создаем список для хранения всей популяции
    # creamos una lista para almacenar toda la población
    
    for _ in range(popSize):
        valido = False
        
        # флаг, который проверяет, что решение содержит все кластеры
        # indicador que verifica que la solución contiene todos los clusters
        
        while not valido:
            individuo = np.random.randint(0, M, size=N)
            
            # генерируем случайное решение длины N со значениями от 0 до M-1
            # generamos una solución aleatoria de longitud N con valores entre 0 y M-1
            
            if len(np.unique(individuo)) == M:
                valido = True
                
                # проверяем, что ни один кластер не пустой
                # verificamos que ningún cluster esté vacío
        
        poblacion.append(individuo)
        
        # добавляем валидное решение в популяцию
        # añadimos la solución válida a la población
    
    return np.array(poblacion)
    
    # преобразуем список в numpy массив и возвращаем
    # convertimos la lista en un array de numpy y lo devolvemos
    

In [5]:
# =========================
# ОЦЕНКА ПОПУЛЯЦИИ
# =========================
# EVALUACIÓN DE LA POBLACIÓN
# =========================

def evaluar_poblacion(poblacion, X, M):
    fitness_vals = []
    # оцениваем каждое разбиение в популяции
    # evaluamos cada partición en la población
    for ind in poblacion:
        fitness_vals.append(fitness(ind, X, M))
    
    return np.array(fitness_vals)

In [6]:
# =========================
# ОБУЧЕНИЕ РАСПРЕДЕЛЕНИЯ
# =========================
# APRENDIZAJE DE LA DISTRIBUCIÓN
# =========================

def aprender_distribucion(mejores, N, M):
    """
    mejores: лучшие решения
    возвращает P(i,j) — вероятность, что элемент i в кластере j
    
    ---
    
    mejores: mejores soluciones
    retorna P(i,j) - probabilidad de que el elemento i esté en el cluster j
    """
    # N - количество точек
    # M - число кластеров
    # N - número de puntos
    # M - número de clusters
    distribucion = np.zeros((N, M))
    # Создаём матрицу вероятностей, где distribucion[i, j] — это вероятность того, что точка i принадлежит кластеру j. 
    # Изначально она заполнена нулями.    # Crear una matriz de probabilidad donde distribucion[i, j] es la probabilidad que el punto i pertenece al grupo j. 
    # Está originalmente lleno de ceros.    # Crear una matriz de probabilidad donde distribucion[i, j] es la probabilidad que el punto i pertenece al grupo j. 
    # Está originalmente lleno de ceros.
    for i in range(N):
    # в каком кластере чаще всего находится точка i
    # # en qué clúster i es más probable que esté
        for ind in mejores:
            distribucion[i, ind[i]] += 1
        
        distribucion[i] /= len(mejores)
    
    return distribucion

In [7]:
# =========================
# ГЕНЕРАЦИЯ НОВОЙ ПОПУЛЯЦИИ
# =========================
# GENERACIÓN DE NUEVA POBLACIÓN
# =========================

def muestrear(distribucion, popSize, N, M):
    # distribucion — матрица вероятностей размера (N x M)
    # distribucion[i][j] = вероятность того, что точка i попадёт в кластер j
    # distribución - matriz de probabilidades de tamaño (N x M)
    # distribución[i][j] = probabilidad de que el punto i caiga en el cluster j

    nueva_poblacion = []  # сюда будем сохранять новые решения
    # aquí guardaremos las nuevas soluciones

    # создаём popSize новых решений (индивидов)
    # creamos popSize nuevas soluciones (individuos)
    for _ in range(popSize):
        
        # создаём пустой индивид (разбиение)
        # длина N — по одному кластеру для каждой точки
        # creamos un individuo vacío (partición)
        # longitud N - un cluster para cada punto
        individuo = np.zeros(N, dtype=int)
        
        # для каждой точки i выбираем кластер
        # para cada punto i elegimos un cluster
        for i in range(N):
            
            # np.random.choice выбирает случайное значение из {0,...,M-1}
            # но НЕ равновероятно, а по заданным вероятностям distribucion[i]
            # np.random.choice selecciona un valor aleatorio de {0,...,M-1}
            # pero NO uniformemente, sino según las probabilidades dadas en distribución[i]
            
            # distribucion[i] — это, например:
            # [0.7, 0.2, 0.1]
            # значит:
            # 70% → кластер 0
            # 20% → кластер 1
            # 10% → кластер 2         
            # # distribucion[i] es, por ejemplo:
            # [0,7, 0,2, 0,1]
            # denota:
            # 70%   clúster 0
            # 20%   grupo 1
            # 10%   grupo 2
            
            individuo[i] = np.random.choice(range(M), p=distribucion[i])
        
        # добавляем готовое решение в популяцию
        # Añade una solución lista para la población
        nueva_poblacion.append(individuo)
    
    return np.array(nueva_poblacion)

In [8]:
# =========================
# ОСНОВНОЙ АЛГОРИТМ EDA
# =========================
# ALGORITMO PRINCIPAL EDA
# =========================
# 🔹 fichero 👉 путь к файлу с данными
# содержит точки (dataset)
# пример: "test.txt"
# 🔹 fichero - ruta al archivo con datos / contiene puntos (dataset) / ejemplo: "test.txt"
# 🔹 popSize
# 👉 размер популяции
# сколько решений (разбиений) храним на каждой итерации
# больше → лучше поиск, но медленнее
# 🔹 popSize - tamaño de la población / cuántas soluciones (particiones) guardamos en cada iteración / más → mejor búsqueda, pero más lento
# 🔹 numPadres
# 👉 число лучших решений для обучения
# сколько "лучших" используем, чтобы построить распределение
# меньше → сильнее давление отбора
# больше → больше разнообразия
# 🔹 numPadres - número de mejores soluciones para entrenar / cuántas "mejores" usamos para construir la distribución / menos → presión de selección más fuerte / más → más diversidad
# 🔹 iteraciones
# 👉 число итераций алгоритма
# сколько раз повторяем цикл EDA
# больше → лучше результат, но дольше
def EDA_clustering(fichero, popSize=50, numPadres=10, iteraciones=50):
    
    # читаем данные
    X, N, K, M = leer_datos(fichero)
    
    # начальная популяция
    # población inicial
    poblacion = inicializar_poblacion(popSize, N, M)
    # лучшее разбиение (индивид)
    # mejor partición (individuo)
    mejor_global = None
    # лучшее значение лучшего разбиения
    # mejor valor de la mejor partición
    # float('inf') — начальное значение, так как задача — минимизация:
    # мы ищем разбиение, минимизирующее сумму квадратов расстояний от точек до центров кластеров
    # float('inf') - valor inicial, ya que la tarea es minimización:
    # buscamos una partición que minimice la suma de cuadrados de distancias de puntos a centros de clusters
    mejor_fitness = float('inf')
    
    for it in range(iteraciones):
        
        # оцениваем
        # ппосчитаем fitness для каждого решения
        # evaluamos / calculamos fitness para cada solución
        fitness_vals = evaluar_poblacion(poblacion, X, M)
        
        # сортируем
        # ordenamos
        orden = np.argsort(fitness_vals)
        poblacion = poblacion[orden]
        # после сортировки fitness_vals[0] — это лучший
        # fitness в текущей популяции
        # fitness_vals[-1] → худший
        # después de ordenar fitness_vals[0] es el mejor fitness / en la población actual / fitness_vals[-1] → peor
        fitness_vals = fitness_vals[orden]
        # мы минимизируем fitness
        # меньше = лучше
        # argsort сортирует по возрастанию
        # minimizamos fitness / menos = mejor / argsort ordena en orden ascendente
        
        # обновляем лучшее
        # actualizamos el mejor
        if fitness_vals[0] < mejor_fitness:
            mejor_fitness = fitness_vals[0]
            mejor_global = poblacion[0].copy()
        
        # выбираем лучших
        # seleccionamos los mejores
        mejores = poblacion[:numPadres]
        
        # получаем распределение
        # obtener distribución
        distribucion = aprender_distribucion(mejores, N, M)
        
        # генерируем новую популяцию
        # # Generar una nueva población

        poblacion = muestrear(distribucion, popSize, N, M)
         
        print(f"Iter {it}: mejor = {fitness_vals[0]:.4f}, media = {np.mean(fitness_vals):.4f}")
    
    print("\n=== RESULTADO FINAL ===")
    print("Mejor fitness:", mejor_fitness)
    print("Mejor partición:", mejor_global)

In [9]:
EDA_clustering("test.txt", 50, 10, 50)

Iter 0: mejor = 0.6900, media = 42.6994
Iter 1: mejor = 0.6900, media = 41.3924
Iter 2: mejor = 0.6900, media = 27.9978
Iter 3: mejor = 0.6900, media = 14.9346
Iter 4: mejor = 0.6900, media = 4.0472
Iter 5: mejor = 0.6900, media = 3.8432
Iter 6: mejor = 0.6900, media = 0.8936
Iter 7: mejor = 0.6900, media = 0.7630
Iter 8: mejor = 0.6900, media = 0.6900
Iter 9: mejor = 0.6900, media = 0.6900
Iter 10: mejor = 0.6900, media = 0.6900
Iter 11: mejor = 0.6900, media = 0.6900
Iter 12: mejor = 0.6900, media = 0.6900
Iter 13: mejor = 0.6900, media = 0.6900
Iter 14: mejor = 0.6900, media = 0.6900
Iter 15: mejor = 0.6900, media = 0.6900
Iter 16: mejor = 0.6900, media = 0.6900
Iter 17: mejor = 0.6900, media = 0.6900
Iter 18: mejor = 0.6900, media = 0.6900
Iter 19: mejor = 0.6900, media = 0.6900
Iter 20: mejor = 0.6900, media = 0.6900
Iter 21: mejor = 0.6900, media = 0.6900
Iter 22: mejor = 0.6900, media = 0.6900
Iter 23: mejor = 0.6900, media = 0.6900
Iter 24: mejor = 0.6900, media = 0.6900
Iter 2